# Commercial Models Inference — GPT & Claude (Thesis Section 4.5)

Runs GPT-4o, GPT-5.2, Claude Opus 4.6, and Claude Sonnet 4 on the test set (510 samples) using the same grading prompt as the fine-tuned model.

**Requirements:**
- `pip install openai anthropic pandas openpyxl scikit-learn`
- `export OPENAI_API_KEY="sk-..."`
- `export ANTHROPIC_API_KEY="sk-ant-..."`

**Note:** This notebook makes live API calls. Results are saved to `GPT_Comparison/` and `Claude_Comparison/`. If results already exist, the notebook skips inference and loads saved results.

**Sections:**
1. Configuration & Shared Prompt
2. Helper Functions
3. Load Test Data
4. GPT Models (GPT-4o, GPT-5.2)
5. Claude Models (Opus 4.6, Sonnet 4)
6. Results Summary

In [ ]:
import os
import re
import json
import time
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score

## 1. Configuration & Shared Prompt

In [ ]:
DATA_PATH = '../Data Statsitical Analysis/Data.xlsx'
GPT_OUTPUT_DIR = '../GPT_Comparison'
CLAUDE_OUTPUT_DIR = '../Claude_Comparison'

GPT_MODELS = ['gpt-4o', 'gpt-5.2']
CLAUDE_MODELS = ['claude-opus-4-6', 'claude-sonnet-4-20250514']

SEED = 42
TEST_QUESTIONS = [4, 8, 18, 19, 21, 22, 25, 31, 45, 55, 57, 68, 69, 75, 80, 81, 84]

PROMPT = """You are an expert AI Teaching Assistant evaluating student responses for accounting questions.

**QUESTION:** {question}

**MODEL ANSWER (Benchmark):** {model_answer}

**STUDENT ANSWER:** {student_answer}

**STEP 1 - CLASSIFY THE ANSWER:**
First, determine the answer category:
- **On-Topic**: Answer addresses the question asked
- **Off-Topic**: Answer has content but doesn't address the question (wrong topic, irrelevant, or meaningless keywords)
- **No Answer**: Answer is blank or empty

**STEP 2 - PROVIDE REASON:**
Select the EXACT text from this list:
- "Valid answer that addresses the question" (Use only for On-Topic)
- "This is an answer to another question, and doesn't relate to model answer" (Use for Off-Topic irrelevant content)
- "Insufficient answer - may have keywords but without any meaning to the required answer" (Use for Off-Topic meaningless content)
- "No answer provided" (Use for No Answer)

**STEP 3 - IF ON-TOPIC, GRADE USING RUBRIC:**
- **Clarity** (0-2): Answer clarity and directness
- **Terminology** (0-2): Correct business/accounting terms
- **Coverage** (0-2): Addresses main points from model answer
- **Accuracy** (0-4): Factual correctness

**CRITICAL:** If Off-Topic or No Answer, ALL scores must be 0.

**OUTPUT FORMAT:** Respond with ONLY valid JSON:
{{
  "off_topic": "[On-Topic|Off-Topic|No Answer]",
  "Off_Topic_Reason": "[reason]",
  "Clarity_Score": X,
  "Terminology_Score": X,
  "Coverage_Score": X,
  "Accuracy_Score": X
}}"""

print(f'Test questions: {TEST_QUESTIONS}')
print(f'GPT models: {GPT_MODELS}')
print(f'Claude models: {CLAUDE_MODELS}')

## 2. Helper Functions

In [ ]:
def clean_text(text):
    """Clean HTML and bullet formatting from text."""
    if not isinstance(text, str):
        return '' if pd.isna(text) else str(text)
    text = re.sub(r'<br\s*/?>', '\n', text, flags=re.IGNORECASE)
    text = re.sub(r'[\u2022\u00A5]\s*\d*\s*', '- ', text)
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    return '\n'.join(lines)


def parse_response(response):
    """Parse LLM JSON response into structured scores."""
    result = {
        'off_topic': 'On-Topic', 'Off_Topic_Reason': '',
        'Clarity_Score': 0, 'Terminology_Score': 0,
        'Coverage_Score': 0, 'Accuracy_Score': 0
    }
    try:
        json_match = re.search(r'\{[^{}]*\}', response, re.DOTALL)
        if json_match:
            parsed = json.loads(json_match.group())
            for key in result:
                if key in parsed:
                    result[key] = parsed[key]
            return result
    except json.JSONDecodeError:
        pass
    # Fallback: regex parsing
    off_topic_match = re.search(r'"off_topic"\s*:\s*"([^"]+)"', response)
    if off_topic_match:
        result['off_topic'] = off_topic_match.group(1)
    reason_match = re.search(r'"Off_Topic_Reason"\s*:\s*"([^"]+)"', response)
    if reason_match:
        result['Off_Topic_Reason'] = reason_match.group(1)
    for key in ['Clarity_Score', 'Terminology_Score', 'Coverage_Score', 'Accuracy_Score']:
        match = re.search(rf'"?{key}"?\s*:\s*(\d+)', response)
        if match:
            result[key] = int(match.group(1))
    return result


def compute_metrics(detailed_results):
    """Compute all evaluation metrics from detailed results list."""
    human_finals, llm_finals = [], []
    human_totals, llm_totals = [], []
    off_topic_correct, on_topic_correct = 0, 0
    false_positive, false_negative = 0, 0

    for r in detailed_results:
        h_final = (r['Human_Clarity'] + r['Human_Terminology'] + r['Human_Coverage'] + r['Human_Accuracy']) / 2
        l_final = (r['LLM_Clarity'] + r['LLM_Terminology'] + r['LLM_Coverage'] + r['LLM_Accuracy']) / 2
        human_finals.append(h_final)
        llm_finals.append(l_final)
        human_totals.append(r['Human_Clarity'] + r['Human_Terminology'] + r['Human_Coverage'] + r['Human_Accuracy'])
        llm_totals.append(r['LLM_Clarity'] + r['LLM_Terminology'] + r['LLM_Coverage'] + r['LLM_Accuracy'])

        if r['Human_OffTopic'] in ['Off-Topic', 'No Answer']:
            if r['LLM_OffTopic'] in ['Off-Topic', 'No Answer']:
                off_topic_correct += 1
            else:
                false_negative += 1
        else:
            if r['LLM_OffTopic'] == 'On-Topic':
                on_topic_correct += 1
            else:
                false_positive += 1

    human_finals = np.array(human_finals)
    llm_finals = np.array(llm_finals)
    diffs = np.abs(human_finals - llm_finals)

    return {
        'acc_1': np.mean(diffs <= 1.0) * 100,
        'acc_05': np.mean(diffs <= 0.5) * 100,
        'mae': np.mean(diffs),
        'qwk': cohen_kappa_score(human_totals, llm_totals, weights='quadratic'),
        'off_topic_acc': (off_topic_correct + on_topic_correct) / len(detailed_results) * 100,
        'false_positive': false_positive,
        'false_negative': false_negative,
    }

## 3. Load Test Data

In [ ]:
df = pd.read_excel(DATA_PATH)
df['Q_num'] = df['ID'].apply(lambda x: int(str(x).split(' ')[0].replace('Q', '')))
test_df = df[df['Q_num'].isin(TEST_QUESTIONS)].copy().reset_index(drop=True)
print(f'Test samples: {len(test_df)}')

## 4. GPT Models (GPT-4o, GPT-5.2)

Requires: `export OPENAI_API_KEY="sk-..."`

In [ ]:
def run_gpt_model(model_name, test_df, output_dir):
    """Run a GPT model on the test set. Skips if results already exist."""
    model_dir = os.path.join(output_dir, model_name.replace('.', '_'))
    os.makedirs(model_dir, exist_ok=True)

    summary_file = os.path.join(model_dir, 'summary.json')
    if os.path.exists(summary_file):
        print(f'  {model_name} already completed — loading saved results.')
        with open(summary_file) as f:
            return json.load(f)

    from openai import OpenAI
    client = OpenAI()
    total = len(test_df)
    detailed_results = []

    # Resume from progress if interrupted
    progress_file = os.path.join(model_dir, 'progress.json')
    start_idx = 0
    if os.path.exists(progress_file):
        with open(progress_file) as f:
            saved = json.load(f)
            detailed_results = saved['results']
            start_idx = len(detailed_results)
            print(f'  Resuming from sample {start_idx}/{total}')

    print(f'Running {model_name} on {total} samples (from {start_idx})...')
    model_start = time.time()

    for i in range(start_idx, total):
        row = test_df.iloc[i]
        user_msg = PROMPT.format(
            question=str(row['Question']).strip(),
            model_answer=clean_text(row['Model Answer']),
            student_answer=str(row['Answer Text']).strip()
        )

        h_off = str(row['off_topic'])
        h_c, h_t, h_v, h_a = int(row['Clarity_Score']), int(row['Terminology_Score']), int(row['Coverage_Score']), int(row['Accuracy_Score'])

        for attempt in range(3):
            try:
                api_params = {
                    'model': model_name,
                    'messages': [{'role': 'user', 'content': user_msg}],
                    'temperature': 0, 'seed': SEED,
                }
                if 'gpt-5' in model_name:
                    api_params['max_completion_tokens'] = 400
                else:
                    api_params['max_tokens'] = 400
                response = client.chat.completions.create(**api_params)
                raw_response = response.choices[0].message.content
                break
            except Exception as e:
                if attempt < 2:
                    print(f'  Retry {attempt+1} for sample {i+1}: {e}')
                    time.sleep(5)
                else:
                    print(f'  FAILED sample {i+1}: {e}')
                    raw_response = '{}'

        llm = parse_response(raw_response)
        m_c, m_t, m_v, m_a = llm['Clarity_Score'], llm['Terminology_Score'], llm['Coverage_Score'], llm['Accuracy_Score']
        h_final = (h_c + h_t + h_v + h_a) / 2
        l_final = (m_c + m_t + m_v + m_a) / 2
        diff = abs(h_final - l_final)

        detailed_results.append({
            'student_id': row['ID'],
            'Human_OffTopic': h_off, 'LLM_OffTopic': llm['off_topic'],
            'OffTopic_Match': h_off == llm['off_topic'],
            'Human_Clarity': h_c, 'LLM_Clarity': m_c,
            'Human_Terminology': h_t, 'LLM_Terminology': m_t,
            'Human_Coverage': h_v, 'LLM_Coverage': m_v,
            'Human_Accuracy': h_a, 'LLM_Accuracy': m_a,
            'Human_Final': h_final, 'LLM_Final': l_final,
            'Diff': diff, 'Within_1.0': diff <= 1.0, 'Within_0.5': diff <= 0.5,
            'raw_response': raw_response,
        })

        if (i + 1) % 50 == 0:
            elapsed = time.time() - model_start
            diffs_so_far = np.array([r['Diff'] for r in detailed_results])
            print(f'  [{i+1}/{total}] ±1Acc: {np.mean(diffs_so_far <= 1.0)*100:.1f}% ({elapsed/60:.1f} min)')
            with open(progress_file, 'w') as f:
                json.dump({'results': detailed_results}, f)

    total_time = time.time() - model_start

    # Save results
    pd.DataFrame(detailed_results).to_csv(os.path.join(model_dir, 'test_results.csv'), index=False)
    metrics = compute_metrics(detailed_results)
    metrics['model'] = model_name
    metrics['total_time_min'] = round(total_time / 60, 1)
    metrics['samples'] = total
    with open(summary_file, 'w') as f:
        json.dump(metrics, f, indent=2)

    if os.path.exists(progress_file):
        os.remove(progress_file)

    return metrics

In [ ]:
# Run GPT models
os.makedirs(GPT_OUTPUT_DIR, exist_ok=True)
gpt_results = []

for model_name in GPT_MODELS:
    metrics = run_gpt_model(model_name, test_df, GPT_OUTPUT_DIR)
    gpt_results.append(metrics)
    print(f'  {model_name}: ±1Acc={metrics["acc_1"]:.2f}% QWK={metrics["qwk"]:.3f} MAE={metrics["mae"]:.3f}')
    print()

## 5. Claude Models (Opus 4.6, Sonnet 4)

Requires: `export ANTHROPIC_API_KEY="sk-ant-..."`

In [ ]:
def run_claude_model(model_name, test_df, output_dir):
    """Run a Claude model on the test set. Skips if results already exist."""
    model_dir = os.path.join(output_dir, model_name.replace('.', '_').replace('-', '_'))
    os.makedirs(model_dir, exist_ok=True)

    summary_file = os.path.join(model_dir, 'summary.json')
    if os.path.exists(summary_file):
        print(f'  {model_name} already completed — loading saved results.')
        with open(summary_file) as f:
            return json.load(f)

    from anthropic import Anthropic
    client = Anthropic()
    total = len(test_df)
    detailed_results = []

    progress_file = os.path.join(model_dir, 'progress.json')
    start_idx = 0
    if os.path.exists(progress_file):
        with open(progress_file) as f:
            saved = json.load(f)
            detailed_results = saved['results']
            start_idx = len(detailed_results)
            print(f'  Resuming from sample {start_idx}/{total}')

    print(f'Running {model_name} on {total} samples (from {start_idx})...')
    model_start = time.time()

    for i in range(start_idx, total):
        row = test_df.iloc[i]
        user_msg = PROMPT.format(
            question=str(row['Question']).strip(),
            model_answer=clean_text(row['Model Answer']),
            student_answer=str(row['Answer Text']).strip()
        )

        h_off = str(row['off_topic'])
        h_c, h_t, h_v, h_a = int(row['Clarity_Score']), int(row['Terminology_Score']), int(row['Coverage_Score']), int(row['Accuracy_Score'])

        for attempt in range(3):
            try:
                response = client.messages.create(
                    model=model_name, max_tokens=400, temperature=0,
                    messages=[{'role': 'user', 'content': user_msg}],
                )
                raw_response = response.content[0].text
                break
            except Exception as e:
                if attempt < 2:
                    print(f'  Retry {attempt+1} for sample {i+1}: {e}')
                    time.sleep(5)
                else:
                    print(f'  FAILED sample {i+1}: {e}')
                    raw_response = '{}'

        llm = parse_response(raw_response)
        m_c, m_t, m_v, m_a = llm['Clarity_Score'], llm['Terminology_Score'], llm['Coverage_Score'], llm['Accuracy_Score']
        h_final = (h_c + h_t + h_v + h_a) / 2
        l_final = (m_c + m_t + m_v + m_a) / 2
        diff = abs(h_final - l_final)

        detailed_results.append({
            'student_id': row['ID'],
            'Human_OffTopic': h_off, 'LLM_OffTopic': llm['off_topic'],
            'OffTopic_Match': h_off == llm['off_topic'],
            'Human_Clarity': h_c, 'LLM_Clarity': m_c,
            'Human_Terminology': h_t, 'LLM_Terminology': m_t,
            'Human_Coverage': h_v, 'LLM_Coverage': m_v,
            'Human_Accuracy': h_a, 'LLM_Accuracy': m_a,
            'Human_Final': h_final, 'LLM_Final': l_final,
            'Diff': diff, 'Within_1.0': diff <= 1.0, 'Within_0.5': diff <= 0.5,
            'raw_response': raw_response,
        })

        if (i + 1) % 50 == 0:
            elapsed = time.time() - model_start
            diffs_so_far = np.array([r['Diff'] for r in detailed_results])
            print(f'  [{i+1}/{total}] ±1Acc: {np.mean(diffs_so_far <= 1.0)*100:.1f}% ({elapsed/60:.1f} min)')
            with open(progress_file, 'w') as f:
                json.dump({'results': detailed_results}, f)

    total_time = time.time() - model_start

    pd.DataFrame(detailed_results).to_csv(os.path.join(model_dir, 'test_results.csv'), index=False)
    metrics = compute_metrics(detailed_results)
    metrics['model'] = model_name
    metrics['total_time_min'] = round(total_time / 60, 1)
    metrics['samples'] = total
    with open(summary_file, 'w') as f:
        json.dump(metrics, f, indent=2)

    if os.path.exists(progress_file):
        os.remove(progress_file)

    return metrics

In [ ]:
# Run Claude models
os.makedirs(CLAUDE_OUTPUT_DIR, exist_ok=True)
claude_results = []

for model_name in CLAUDE_MODELS:
    metrics = run_claude_model(model_name, test_df, CLAUDE_OUTPUT_DIR)
    claude_results.append(metrics)
    print(f'  {model_name}: ±1Acc={metrics["acc_1"]:.2f}% QWK={metrics["qwk"]:.3f} MAE={metrics["mae"]:.3f}')
    print()

## 6. Results Summary

In [ ]:
all_results = gpt_results + claude_results

print('='*65)
print('COMMERCIAL MODELS — RESULTS SUMMARY')
print('='*65)
print(f'{"Model":<30} {"±1.0 Acc":>10} {"±0.5 Acc":>10} {"MAE":>8} {"QWK":>8} {"OT Acc":>10}')
print('-'*65)
for m in all_results:
    print(f'{m["model"]:<30} {m["acc_1"]:>9.2f}% {m["acc_05"]:>9.2f}% {m["mae"]:>8.3f} {m["qwk"]:>8.3f} {m["off_topic_acc"]:>9.2f}%')
print('-'*65)

print(f'\nResults saved to:')
print(f'  GPT:    {GPT_OUTPUT_DIR}/')
print(f'  Claude: {CLAUDE_OUTPUT_DIR}/')